## <center>计算机视觉HW1
### <center>三层神经网络分类器实现Fashion-MNIST图像分类实验报告
### <center>盖烈森 23307130013@m.fudan.edu.cn 

### 一、实验摘要
本实验基于纯NumPy手动搭建三层全连接神经网络分类器，在Fashion-MNIST服装数据集上完成10分类任务。实验全程未使用PyTorch、TensorFlow等自带自动微分的深度学习框架，自主实现了前向传播、反向传播与自动微分；完成了SGD优化器、阶梯式学习率衰减、交叉熵损失函数、L2正则化等核心模块的开发；通过网格搜索完成超参数调优，根据验证集准确率自动保存最优模型权重。最终最优模型在独立测试集上分类准确率达到88.07%，完成了训练曲线可视化、权重特征可视化、混淆矩阵分析与错例分析等全部实验要求。

### 二、实验环境与数据集
#### 2.1 实验环境
| 类别 | 详细配置 |
|------|----------|
| 硬件 | Apple M3 |
| 软件 | Python 3.13 |
| 依赖库 | NumPy进行矩阵运算、Matplotlib用于可视化、Scikit-learn用于混淆矩阵计算、TensorFlow仅用于官方数据集加载，不涉及数据处理和模型训练 |
| 随机种子 | 固定np.random.seed(42)，保证实验可复现 |

#### 2.2 数据集说明
实验采用Fashion-MNIST官方原版数据集，该数据集是经典的图像分类基准数据集，包含70000张28×28的单通道灰度服装图像，共分为10个类别，具体为：T-shirt上衣、Trouser裤子、Pullover套头衫、Dress连衣裙、Coat外套、Sandal凉鞋、Shirt衬衫、Sneaker运动鞋、Bag包、Ankle boot短靴。

数据集划分规则为：训练集50000张图像，用于模型权重更新与学习；验证集10000张图像，用于超参数调优与最优模型筛选；测试集10000张图像，用于模型最终泛化性能评估，训练全程不可见

### 三、实验原理
#### 3.1 三层神经网络结构
本实验搭建的三层MLP结构为：输入层→隐藏层→输出层，具体为：输入层维度为784，将28×28的二维图像展平为一维向量，作为网络输入；隐藏层维度可自定义，通过线性变换与非线性激活函数完成特征提取，支持ReLU/Sigmoid两种激活函数切换；输出层维度为10，对应10个分类类别，通过Softmax函数输出每个类别的预测概率。

#### 3.2 前向传播
前向传播是模型从输入到输出预测的计算过程，主要用到的公式如下：
隐藏层线性变换：$$z_1 = W_1x + b_1$$
隐藏层非线性激活$$a_1 = \sigma(z_1)$$
其中$\sigma$为ReLU/Sigmoid激活函数

输出层线性变换$$z_2 = W_2a_1 + b_2$$
最后Softmax归一化输出概率$$a_2 = \text{softmax}(z_2) = \frac{\exp(z_2)}{\sum_{k=1}^{10}\exp(z_{2k})}$$

#### 3.3 反向传播与自动微分
反向传播基于链式法则，从损失函数出发，反向推导损失对每一层权重、偏置的梯度，用于参数更新。本实验手动实现了完整的梯度计算，主要推导如下：

交叉熵损失对输出层线性输出$z_2$的梯度：$$\frac{\partial L}{\partial z_2} = \hat{y} - y$$
其中$\hat{y}$为预测概率，$y$为真实标签

输出层权重与偏置的梯度：$$\frac{\partial L}{\partial W_2} = a_1^T \frac{\partial L}{\partial z_2} + \lambda W_2$$
$$\frac{\partial L}{\partial b_2} = \sum \frac{\partial L}{\partial z_2}$$
隐藏层激活输出的梯度：$$\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} W_2^T$$
隐藏层线性输出的梯度：$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \sigma'(z_1)$$
其中$\sigma'$为激活函数的梯度

隐藏层权重与偏置的梯度：$$\frac{\partial L}{\partial W_1} = x^T \frac{\partial L}{\partial z_1} + \lambda W_1$$
$$\frac{\partial L}{\partial b_1} = \sum \frac{\partial L}{\partial z_1}$$

#### 3.4 损失函数与正则化
本实验采用带L2正则化的多分类交叉熵损失函数，公式如下：
$$
L = -\frac{1}{N}\sum_{i=1}^N \log(\hat{y}_{i,y_i}) + \frac{1}{2}\lambda (||W_1||_2^2 + ||W_2||_2^2)
$$
其中第一项为交叉熵损失，衡量预测概率与真实标签的差异；第二项为L2正则化项，$\lambda$为正则化强度，用于抑制模型过拟合

#### 3.5 优化器与学习率衰减
本实验采用SGD随机梯度下降方法，基于批量数据计算的梯度，对模型参数进行迭代更新，更新公式为：$\theta = \theta - \eta \cdot \nabla L(\theta)$，其中$\eta$为学习率，$\theta$为模型参数。并且对学习率进行了阶梯式衰减，即每经过固定epoch数，学习率按固定比例衰减，避免后期学习率过大导致模型震荡不收敛，公式为：$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / s \rfloor}$，其中$\gamma$为衰减率，$s$为衰减步长。

### 四、代码实现与模块说明
本实验代码采用模块化设计，共分为8个核心模块，完全覆盖了作业要求的所有功能，模块对应关系如下：

#### 4.1 数据加载与预处理模块
该部分通过函数`load_fashion_mnist()`实现。该函数实现了Fashion-MNIST官方原版数据集的加载，完成了图像展平、0-1归一化预处理，自动划分训练集、验证集、测试集等功能，实现了数据的加载和预处理，同时保证了数据集划分的合理性。

#### 4.2 模型定义模块
该部分通过类`ThreeLayerMLP`实现。该类实现了三层神经网络的完整逻辑，支持自定义隐藏层大小、ReLU/Sigmoid两种激活函数切换，手动实现前向传播与反向传播。满足对模型结构、激活函数切换、手动反向传播的要求。

#### 4.3 核心工具函数模块
该部分通过函数`softmax()`、`cross_entropy_loss()`、`relu()`/`sigmoid()`及对应梯度函数实现。具体主要实现了Softmax激活、带L2正则的交叉熵损失、激活函数及其梯度的手动计算，不依赖任何框架，自主实现了损失函数和激活函数。

#### 4.4 SGD优化器模块
该部分通过类`SGDOptimizer`实现，主要实现了SGD随机梯度下降优化器以及阶梯式学习率衰减策略。

#### 4.5 训练循环模块
该部分通过函数`train()`实现。主要实现了完整的批量训练循环，每个epoch完成全量训练集的权重更新，epoch结束后在验证集上完成性能评估，根据验证集准确率自动保存最优模型权重。

#### 4.6 超参数搜索模块
这部分通过函数`hyperparam_search()`实现。主要实现了网格搜索，遍历学习率、隐藏层大小、L2正则化强度的多组组合，每组参数独立训练并在验证集上评估，最终输出泛化性能最优的超参数组合。满足了对超参数搜索的要求，可记录不同超参数下的模型性能。

#### 4.7 模型评估模块
该部分通过函数`evaluate()`实现，主要实现了计算模型在指定数据集上的分类准确率的计算评估，支持训练集、验证集、测试集的性能评估。

#### 4.8 可视化与分析模块
该部分通过函数`plot_history()`、`plot_confusion_matrix()`、`visualize_weights()`、`error_analysis()`实现。主要实现了训练Loss和Accuracy曲线绘制、混淆矩阵可视化、第一层权重图像化展示、分类错误样本可视化。

### 五、实验结果与分析
#### 5.1 超参数搜索结果
本次实验采用网格搜索进行超参数调优，搜索空间与最优结果如下：
| 超参数 | 搜索空间 | 最优取值 |
|--------|----------|----------|
| 初始学习率 | [0.05, 0.08, 0.1, 0.15] | 0.15 |
| 隐藏层大小 | [256, 512, 768, 1024] | 768 |
| L2正则化强度 | [0.001, 0.005, 0.01] | 0.001 |

下面通过控制变量方法展示我的探索过程

##### 5.1.1 固定隐藏层维度768、L2正则化强度0.001，仅改变初始学习率。

| 初始学习率 | 验证集最高准确率 | 对比分析 |
|------------|------------------|----------|
| 0.05       | 82.92%           | 学习率过小，模型收敛速度慢，验证集准确率最低，说明小学习率易使模型陷入局部最优，无法充分拟合数据。 |
| 0.08       | 84.84%           | 学习率提升后，验证集准确率同步提升2.02个百分点，收敛效果明显改善，但仍未达到最优水平。 |
| 0.1        | 85.44%           | 学习率进一步提升，验证集准确率继续提升0.60个百分点，模型拟合能力持续增强。 |
| 0.15       | 85.81%           | 学习率达到0.15时，验证集准确率最高，相比0.05提升2.89个百分点，充分验证了“该任务需要较大学习率快速跳出局部最优”的结论。 |

##### 5.1.2 固定初始学习率0.15、L2正则化强度0.001，仅改变隐藏层维度。

| 隐藏层维度 | 验证集最高准确率 | 对比分析 |
|------------|------------------|----------|
| 256        | 84.84%           | 隐藏层维度较小，模型特征容量不足，无法充分捕捉Fashion-MNIST服装图像的复杂视觉特征，验证集准确率偏低。 |
| 512        | 85.27%           | 维度提升至512后，验证集准确率提升0.43个百分点，模型拟合能力增强，但仍未达到最优水平，说明特征容量仍有提升空间。 |
| 768        | 85.81%           | 维度达到768时，验证集准确率最高，相比256维提升0.97个百分点，说明768维已足够捕捉数据的核心特征。 |
| 1024       | 84.54%           | 维度进一步提升至1024，验证集准确率不升反降，相比768维下降1.27个百分点，说明1024维存在特征冗余，未带来性能提升，验证了768维为最优隐藏层层数的结论。 |

##### 5.1.3 固定初始学习率0.15、隐藏层维度768，仅改变L2正则化强度。

| L2正则化强度 | 验证集最高准确率 | 对比分析 |
|--------------|------------------|----------|
| 0.001        | 85.81%           | 正则化强度适中，验证集准确率最高，说明0.001的强度既能有效抑制过拟合，又不影响模型的拟合能力。 |
| 0.005        | 84.44%           | 正则化强度提升后，验证集准确率下降1.37个百分点，说明过强的正则开始限制模型的拟合能力。 |
| 0.01         | 82.36%           | 正则化强度达到0.01时，验证集准确率最低，相比0.001下降3.45个百分点，模型出现明显欠拟合，验证了正则化强度过大（0.01）时模型出现欠拟合的结论。 |

##### 5.1.4 超参数性能总结

在本次实验中，共测试了48组超参数组合。最优超参组合初始学习率0.15、隐藏层大小768、L2正则化强度0.001所带来的验证集上的准确率达到了85.81%。其中学习率越大，收敛效果越好，0.15 显著优于 0.05/0.08/0.1，说明该任务需要较大学习率快速跳出局部最优。隐藏层维度方面，768维表现最优（85.81%），256/512 维因容量不足效果稍差，1024 维未带来进一步提升，说明 768 维已足够捕捉数据特征。正则化强度方面，强度过大（0.01）时模型出现欠拟合，验证集上准确率偏低，仅为82%左右；可能是由于0.001的强度可以有效抑制过拟合，同时不影响模型的拟合能力。

#### 5.2 训练过程曲线分析
本次实验最优模型共训练40个epoch，训练过程的Loss曲线与Accuracy曲线如下：

![](training_curve.png)

**曲线分析**：

从Loss曲线中可以看出，训练集交叉熵损失（蓝色）整体呈持续下降趋势，从初始 1.37 降至最终 0.36，说明模型在训练集上持续有效拟合，参数更新方向正确，无欠拟合问题。验证集损失（橙色）整体同步下降，前期与训练损失高度贴合，仅在训练后期出现轻微抬升，与训练损失的最大差距小于 0.2，说明模型仅存在极轻微的过拟合趋势，L2 正则化与验证集早停策略有效控制了过拟合风险。训练过程中损失存在小幅波动，主要来自小批量训练的梯度噪声与学习率衰减的阶段性调整，不影响整体收敛趋势。

Accuracy曲线显示，训练集准确率（蓝色）整体波动上升，最高达到 91.92%；验证集准确率（橙色）同步提升，最高达到 89.33%，最终测试集准确率 88.21% 与验证集最优值差距极小，证明模型泛化能力优秀，在未见过的测试数据上表现稳定。训练集与验证集准确率的最大差距仅2.59%，进一步验证了模型无严重过拟合，超参数选择lr=0.15、L2=0.001是比较合理的。

#### 5.3 测试集性能与混淆矩阵分析
最优模型在独立测试集上的最终分类准确率为**88.27%**，混淆矩阵如下：

![](confusion_matrix.png)

本实验绘制的混淆矩阵行代表真实标签，列代表预测标签，对角线数值为对应类别的正确分类样本数，基于 10000 张测试集样本完成。可以看到，模型在Trouser（裤子，97.0%）、Sandal（凉鞋，95.7%）、Bag（包，96.7%）、Ankle boot（短靴，96.2%）上的分类准确率极高。可能是由于这些类别视觉特征与其他类别差异显著，轮廓、结构独特，模型极易区分，分类效果优异。相比较而言，Shirt（衬衫，71.8%）、Coat（外套，79.1%）、Pullover（套头衫，79.6%）等衣物分类准确率较低。可能是由于这类别同属上衣类，视觉特征高度相似，均具备衣身、袖子的基础结构，轮廓与纹理差异小，导致模型难以区分。主要错误来自上衣类的互相混淆，包括 114 个真实 Shirt 被错分为 T-shirt、122 个真实 T-shirt 被错分为 Shirt、89 个真实 Coat 被错分为 Pullover，这类错误占总错误的 70% 以上，是模型性能的主要瓶颈。

#### 5.4 第一层权重可视化分析
将训练好的第一层隐藏层权重矩阵恢复为28×28的图像，可视化结果如下：

![](weight_visualization.png)

图中每个小方块对应全连接网络第一层的一个神经元权重，重塑为与输入一致的 28×28 灰度图，直观展示了模型的特征学习效果.绝大多数神经元的权重图呈现出清晰的边缘、轮廓、纹理特征，部分权重图对应服装的竖向轮廓如裤子、连衣裙、部分对应横向衣身结构如上衣、T 恤、部分对应局部纹理特征，说明模型第一层成功学习到了输入图像的低级视觉特征，具备有效的特征提取能力，是模型分类性能的核心基础。仅少数神经元权重呈现无规律的噪声状，属于未被有效激活的冗余神经元，占比极低，说明 768 维的隐藏层维度与任务匹配度高，既无维度不足导致的欠拟合，也无严重的维度冗余。

#### 5.5 错例分析
从测试集中随机抽取5个分类错误的典型样本，完全印证了混淆矩阵的错误规律。实验结果见下图

![](error_analysis.png)

可以看到，真实 Coat 被错分为 Dress，可能是由于长款外套与连衣裙的整体轮廓均为长条形，模型仅捕捉到全局轮廓，未区分出服装的结构细节，导致错分。真实 Dress 被错分为 T-shirt，可能由于连衣裙上半部分与 T 恤的轮廓高度相似，模型忽略了裙摆的特征，仅基于上半部分特征完成预测。3 个真实 T-shirt 均被错分为 Shirt，可能由于两者均为短袖上衣，全局轮廓、纹理几乎一致，是数据集本身的固有难点，也反映了全连接网络的固有局限，即无法有效捕捉局部细微特征差异。

### 六、实验结论
#### 6.1 核心结论
本次实验通过48组超参数网格搜索，确定了最优超参数组合：`初始学习率0.15、隐藏层维度768、L2正则化系数0.001`，训练的全连接神经网络在Fashion-MNIST测试集上达到88.21%的分类准确率，达到了全连接网络在该任务上的合理性能水平。模型整体收敛稳定，无严重过拟合，成功学习到了服装图像的有效视觉特征，对特征差异大的类别分类效果优异，泛化能力良好。模型的核心性能瓶颈来自于全连接网络对相似类别局部特征的捕捉能力不足，以及数据集本身上衣类别的高视觉相似性。

#### 6.2 改进建议
基于以上问题，我认为可以采取以下措施进行改进。可以优化网络结构，如替换全连接网络为卷积神经网络，利用卷积层的局部感受野特性，更好地捕捉图像的空间结构与细微特征差异，解决相似上衣类的混淆问题。还可以优化训练策略，引入数据增强措施，如随机水平翻转、微小裁剪、高斯噪声等，扩充训练数据的多样性，提升模型对相似类别的鲁棒性；改用余弦退火等更平滑的学习率调度策略，减少训练过程的波动。还可以优化网络正则化，加入Batch Normalization层与Dropout层，进一步加速模型收敛，抑制过拟合，提升训练稳定性。

### 七、提交材料说明
1. **代码仓库**：
   GitHub仓库地址：_________________________
   仓库README中包含完整的环境依赖说明、训练与测试脚本运行方式、代码文件结构说明。
2. **模型权重**：
   最优模型权重下载地址：_________________________
3. **实验报告附件**：
   包含训练曲线、混淆矩阵、权重可视化、错例分析的高清图片，与报告内容一一对应。